# Phase 2: Feature Engineering & Modeling
## 1. Import Libraries

In [8]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

## 2. Load Cleaned Data

In [9]:
df = pd.read_parquet('../datasets/cleaned_data.parquet')
print(f"Loaded shape: {df.shape}")

# Handle missing values and infs (sometimes they persist from phase 1)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"Shape after dropping NaNs/Infs: {df.shape}")

# Memory Optimization: Ensure all numerical features are float32
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = df[col].astype(np.float32)
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = df[col].astype(np.int32)

print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Loaded shape: (2522362, 71)
Shape after dropping NaNs/Infs: (2520798, 71)
Memory usage: 865.1 MB


## 3. Feature Selection & Engineering

In [10]:
# Encode Labels
le = LabelEncoder()
df['Label_Encoded'] = le.fit_transform(df['Label'])
joblib.dump(le, '../models/label_encoder.pkl')
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Feature Selection (Keeping ~18 important features based on standard CIC-IDS2017 practices)
selected_features = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean', 'Packet Length Std', 
    'SYN Flag Count', 'ACK Flag Count', 'FIN Flag Count', 'RST Flag Count', 
    'PSH Flag Count', 'URG Flag Count'
]

# Ensure selected features exist in columns
available_cols = df.columns.tolist()
features_to_use = [f for f in selected_features if f in available_cols]
print(f"Using {len(features_to_use)} features.")

X = df[features_to_use]
y = df['Label_Encoded']
joblib.dump(features_to_use, '../models/selected_features.pkl')

Classes mapping: {'BENIGN': np.int64(0), 'Bot': np.int64(1), 'DDoS': np.int64(2), 'DoS GoldenEye': np.int64(3), 'DoS Hulk': np.int64(4), 'DoS Slowhttptest': np.int64(5), 'DoS slowloris': np.int64(6), 'FTP-Patator': np.int64(7), 'Heartbleed': np.int64(8), 'Infiltration': np.int64(9), 'PortScan': np.int64(10), 'SSH-Patator': np.int64(11), 'Web Attack - Brute Force': np.int64(12), 'Web Attack - Sql Injection': np.int64(13), 'Web Attack - XSS': np.int64(14)}
Using 18 features.


['../models/selected_features.pkl']

## 4. Scalable Modeling Pipeline
**Strategy:**
1. Split data 80/20 (stratified)
2. Sample 150k rows from training set for lightweight hyperparameter tuning
3. Use `RandomizedSearchCV` with `n_iter=3`, `cv=2`, `n_jobs=1` (memory-safe)
4. Retrain best configuration on FULL training data
5. Evaluate on held-out test set only

**Models (scalable only):**
- Logistic Regression (with StandardScaler)
- Naive Bayes (with StandardScaler)
- Random Forest (no scaling needed)

**Removed:** SVM (O(n^2+)), KNN (high memory + slow inference)

In [11]:
# ============================================================
# STEP 1: Train/Test Split (stratified, reproducible)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# ============================================================
# STEP 2: Create a SMALL sample for hyperparameter tuning
#          (avoids running CV on 2M+ rows)
# ============================================================
SAMPLE_SIZE = min(150_000, len(X_train))

X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    train_size=SAMPLE_SIZE,
    random_state=42,
    stratify=y_train
)
print(f"Tuning sample: {X_tune.shape}")

Train: (2016638, 18), Test: (504160, 18)
Tuning sample: (150000, 18)


In [12]:
# ============================================================
# STEP 3: Define model pipelines (SCALABLE ONLY)
#   - StandardScaler for LR and NB (distance/distribution-based)
#   - NO scaler for Random Forest (tree-based, scale-invariant)
#   - Removed: SVM (O(n^2+)), KNN (high memory)
# ============================================================
pipelines = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),

    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GaussianNB())
    ]),

    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(random_state=42, n_jobs=1))
    ])
}

# ============================================================
# STEP 4: Hyperparameter search spaces (lightweight)
# ============================================================
param_distributions = {
    "Logistic Regression": {
        "clf__C": [0.01, 0.1, 1, 10],
    },

    "Naive Bayes": {},  # No tuning needed

    "Random Forest": {
        "clf__n_estimators": [50, 100, 200],
        "clf__max_depth": [10, 20, 30],
        "clf__min_samples_split": [2, 5, 10]
    }
}

# ============================================================
# STEP 5: Hyperparameter Tuning on SAMPLE subset only
# ============================================================
best_params = {}

for name in pipelines:
    print(f"\n[TUNING] {name} on {SAMPLE_SIZE:,} samples...")

    if not param_distributions[name]:
        # No tuning needed (e.g., Naive Bayes)
        pipelines[name].fit(X_tune, y_tune)
        best_params[name] = {}
        print(f"  -> No hyperparameters to tune.")
        continue

    search = RandomizedSearchCV(
        pipelines[name],
        param_distributions[name],
        n_iter=3,          # Lightweight: only 3 combinations
        cv=2,              # 2-fold: fast + still validates
        scoring="accuracy",
        n_jobs=1,          # STRICT: single-threaded to prevent RAM explosion
        random_state=42,
        verbose=1
    )

    search.fit(X_tune, y_tune)
    best_params[name] = search.best_params_
    print(f"  -> Best params: {search.best_params_}")
    print(f"  -> CV accuracy (on sample): {search.best_score_:.4f}")

print("\n" + "=" * 60)
print("[DONE] Tuning complete. Best parameters found:")
for name, params in best_params.items():
    print(f"  {name}: {params if params else '(default)'}")


[TUNING] Logistic Regression on 150,000 samples...
Fitting 2 folds for each of 3 candidates, totalling 6 fits
  -> Best params: {'clf__C': 0.1}
  -> CV accuracy (on sample): 0.9412

[TUNING] Naive Bayes on 150,000 samples...
  -> No hyperparameters to tune.

[TUNING] Random Forest on 150,000 samples...
Fitting 2 folds for each of 3 candidates, totalling 6 fits
  -> Best params: {'clf__n_estimators': 100, 'clf__min_samples_split': 5, 'clf__max_depth': 20}
  -> CV accuracy (on sample): 0.9964

[DONE] Tuning complete. Best parameters found:
  Logistic Regression: {'clf__C': 0.1}
  Naive Bayes: (default)
  Random Forest: {'clf__n_estimators': 100, 'clf__min_samples_split': 5, 'clf__max_depth': 20}


In [13]:
# ============================================================
# STEP 6: Retrain on FULL training data with best hyperparams
# ============================================================
print("Retraining each model on FULL training data...\n")

final_models = {}
results = {}

for name, pipeline in pipelines.items():
    print(f"Training {name}...")

    # Build a fresh pipeline with best params
    fresh_pipeline = Pipeline(pipeline.steps)

    # Apply best hyperparameters
    if best_params[name]:
        fresh_pipeline.set_params(**best_params[name])

    # Train on FULL training set (no CV, no tuning)
    fresh_pipeline.fit(X_train, y_train)

    # Evaluate on test set
    y_pred = fresh_pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    final_models[name] = fresh_pipeline
    results[name] = acc

    print(f"  [OK] Test Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("-" * 60)

Retraining each model on FULL training data...

Training Logistic Regression...
  [OK] Test Accuracy: 0.9405
              precision    recall  f1-score   support

           0       0.96      0.97      0.97    419012
           1       1.00      0.01      0.02       390
           2       0.83      0.64      0.72     25603
           3       0.95      0.51      0.67      2057
           4       0.97      0.89      0.93     34569
           5       0.70      0.04      0.08      1046
           6       0.00      0.00      0.00      1077
           7       0.00      0.00      0.00      1186
           8       0.00      0.00      0.00         2
           9       0.43      0.43      0.43         7
          10       0.69      0.99      0.81     18139
          11       0.00      0.00      0.00       644
          12       0.00      0.00      0.00       294
          13       0.00      0.00      0.00         4
          14       0.00      0.00      0.00       130

    accuracy             

In [14]:
# ============================================================
# STEP 7: Select and Save the Best Model
# ============================================================
best_model_name = max(results, key=results.get)
best_model = final_models[best_model_name]

print(f"[BEST] Best model: {best_model_name}")
print(f"       Accuracy:   {results[best_model_name]:.4f}")

# Summary table
print(f"\n{'Model':<25} {'Accuracy':>10}")
print("-" * 35)
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    marker = " <-- best" if name == best_model_name else ""
    print(f"{name:<25} {acc:>10.4f}{marker}")

# Save all artifacts
os.makedirs('../models', exist_ok=True)

# Save best model pipeline (includes scaler if applicable)
model_path = f"../models/{best_model_name.lower().replace(' ', '_')}_best.pkl"
joblib.dump(best_model, model_path)
print(f"\n[SAVE] Best model saved to {model_path}")

# Also save a standalone scaler (for Notebook 03 simulation compatibility)
# Extract from the best model if it has one, otherwise fit a new one
if 'scaler' in dict(best_model.named_steps):
    scaler = best_model.named_steps['scaler']
else:
    # If best model is RF (no scaler), fit a scaler anyway for simulation
    scaler = StandardScaler()
    scaler.fit(X_train)
joblib.dump(scaler, '../models/scaler.pkl')
print("[SAVE] Scaler saved to ../models/scaler.pkl")

# Save all models (for comparison in deployment)
for name, model in final_models.items():
    path = f"../models/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, path)
    print(f"[SAVE] {name} saved to {path}")

[BEST] Best model: Random Forest
       Accuracy:   0.9976

Model                       Accuracy
-----------------------------------
Random Forest                 0.9976 <-- best
Logistic Regression           0.9405
Naive Bayes                   0.3218



[SAVE] Best model saved to ../models/random_forest_best.pkl
[SAVE] Scaler saved to ../models/scaler.pkl
[SAVE] Logistic Regression saved to ../models/logistic_regression.pkl
[SAVE] Naive Bayes saved to ../models/naive_bayes.pkl
[SAVE] Random Forest saved to ../models/random_forest.pkl
